# When the law keeps children in school

**A guided investigation · Chapter 6 · 30–45 minutes · No Python experience needed**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cmg777/intro2causal/blob/main/notebooks_colab/explore-childlabor_clean.ipynb)

> **The question**  
> How do schooling and wages vary across cohorts exposed to different schooling requirements?

School attendance is shaped by family circumstances as well as legal requirements. Acemoglu and Angrist studied historical schooling laws to investigate education and earnings. This teaching file is already aggregated: a row represents a birth-state, birth-year, and census-year cell, with average schooling and log weekly wages. The law indicators distinguish required-schooling categories associated with child-labor restrictions; they must not be interpreted as ages for entering school. This distinction matters because an appealing story can be wrong even when the code runs. We will compare cohort patterns and law categories while keeping the aggregate observation unit visible. States and cohorts differ in many ways besides laws, so these charts are a starting point for asking what variation might support an instrument, not a direct estimate of a law’s causal effect.

### By the end, you will be able to…

- Read aggregate cells carefully and explore how legal requirements might create schooling variation.
- Read and modify three interactive charts using short Python examples.
- Explain one reason a descriptive pattern alone cannot settle this case’s causal question.

### Before you see the evidence

Would states with stricter schooling requirements necessarily be comparable in other ways?

Write two sentences now. You will return to them at the end.

**Your route:** meet the data → inspect three views → modify and try → ask the causal question.

## Your first few minutes in Colab · 3 minutes

You do not need prior Python experience. A **cell** is one block of text or code. Read the text, then press the **▶** beside the next code cell (or Shift+Enter). Its output appears underneath. Run cells in order: later cells use tables created earlier.

Choose **File → Save a copy in Drive** if you want to keep your changes. If a name is “not defined,” run the earlier setup cells again. **Runtime → Restart session and run all** gives you a fresh start. An error does not mean you have damaged the data: this notebook only reads it.

Keep a short learning log in a text cell. Before each chart, make a prediction. Afterward, write one observation with a number and one question that the chart cannot answer.

**Chart controls:** hover for values, click legend labels to hide/show a group, drag to zoom, and double-click to reset. A chart’s menu also offers a reset/autoscale control.

The setup installs Plotly Express if needed. `import` makes a library available; `pd` and `px` are short names for pandas (tables) and Plotly Express (charts).

In [ ]:
%pip install -q "plotly>=6,<7"

In [ ]:
import pandas as pd
import plotly.express as px
from IPython.display import display

# One consistent chart style; no plotting experience needed.
px.defaults.template = "plotly_white"
px.defaults.color_continuous_scale = "Blues"
px.defaults.color_discrete_sequence = ["#2563eb", "#7c3aed", "#059669", "#64748b"]

## Meet the data · 5 minutes

**One row:** Birth-state × birth-year × census-year cell.  
**Stored size:** 2,394 rows × 10 columns.

| Column | What it records |
|---|---|
| `yob` | Year of birth; two-digit 1930s codes in qob_clean, full years in childlabor_clean. |
| `indEduc` | Cell mean of highest completed grade (years). |
| `lnwkwage` | Cell mean of log weekly wages. |
| `cl7` | Indicator for child-labor required-schooling category of 7 years; not entry age. |
| `cl8` | Indicator for child-labor required-schooling category of 8 years; not entry age. |
| `cl9` | Indicator for child-labor required-schooling category of at least 9 years; not entry age. |

cl7, cl8 and cl9 identify required-schooling categories (7, 8, and 9 or more years), not school-entry ages. weight is a source sample-line analysis weight; n is a stored cell count. Equal-cell summaries are explicitly unweighted.

`read_csv` downloads a table. We call it `df`. `head(5)` shows just five rows, not a separate sample used for the analysis. Square brackets select columns by their names.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/cmg777/intro2causal/main/data/ch6/childlabor_clean.csv"
df = pd.read_csv(DATA_URL)
print(f"Loaded {len(df):,} rows and {len(df.columns)} columns")
display(df[['yob', 'indEduc', 'lnwkwage', 'cl7', 'cl8', 'cl9']].head(5))

### Pause: what do the rows mean?

Choose one displayed row. Translate it into a sentence using the variable guide. Which values are categories or identifiers? Which are quantities? A numeric storage type does not necessarily make a code a meaningful measurement.

The next table checks types, missing values, and the number of distinct values for the key columns. `isna()` marks missing cells; `sum()` counts them. Missing is not zero. Ask which comparison would lose a row if one of its required values were missing.

In [ ]:
key_columns = ['yob', 'indEduc', 'lnwkwage', 'cl7', 'cl8', 'cl9']
checks = pd.DataFrame({"storage_type": df[key_columns].dtypes.astype(str),
                       "missing_rows": df[key_columns].isna().sum(),
                       "distinct_values": df[key_columns].nunique()})
display(checks)
print("The charts below omit missing values required for each plotted quantity.")

### A small summary before plotting

`describe()` summarizes numeric columns: `count` counts available values, `mean` is the average, and `50%` is the median. The median splits ordered values into two halves. Averages of identifiers or arbitrary category codes are not useful descriptions, even if Python calculates them.

For category-like key variables, the second table counts values, including missing entries. Check these labels before interpreting a chart.

In [ ]:
numeric_keys = df[key_columns].select_dtypes(include="number").columns
if len(numeric_keys):
    display(df[numeric_keys].describe().round(2))
for column in key_columns:
    if df[column].nunique() <= 8:
        display(df[column].value_counts(dropna=False).rename("Rows").to_frame())

## Read the chart code · 2 minutes

`px.bar`, `px.scatter`, `px.line`, and `px.box` choose a chart type. `x` and `y` name columns for the axes; `color` distinguishes groups or values. `labels` translates column names for readers. `fig.show()` displays the result.

A table transformation and a plot answer different questions: first decide what each plotted row represents, then choose how to display it. Before running each chart, predict what you expect to see.

In [ ]:
# Display labels: run once; you do not need to memorize this mapping.
labels = {'ability_group': 'Ability group',
 'census': 'Census year',
 'group': 'State group',
 'indEduc': 'Cell mean schooling (years)',
 'lnwkwage': 'Cell mean log weekly wages',
 'n': 'Students in score bin',
 'offer': 'Scholarship assignment',
 'plan': 'Assigned insurance plan',
 'sob': 'Birth-state code',
 'state_name': 'Simulated state',
 'year': 'Year',
 'yob': 'Birth year'}

## Visual investigation 1 · Follow generations

**Predict:** How does average schooling vary across birth cohorts in these cells?

We group by birth year and average the cell means. This gives each available cell equal weight; it is not a population-weighted estimate. The census waves can cover different cohorts.

In [ ]:
cohorts = df.groupby(["yob", "year"], as_index=False)["indEduc"].mean()
cohorts["census"] = cohorts["year"].astype(str)
fig = px.line(cohorts, x="yob", y="indEduc", color="census", markers=True, labels={"yob": "Birth year", "indEduc": "Mean schooling across cells (years)", "census": "Census year"}, title="Schooling across cohorts: equal cell weights")
fig.show()

**Explore the chart:** Toggle census years to compare the covered cohorts. Do not mistake the census year for a birth year.

**Write before reading on:** Describe one pattern with its unit and comparison group. Then give one possible explanation.

<details>
<summary>Interpretation checkpoint</summary>

Cohort changes can reflect many historical developments. They cannot be attributed solely to schooling laws.

</details>

## Visual investigation 2 · Compare the law categories

**Predict:** Do cells in different requirement categories have similar schooling?

The indicators classify child-labor schooling requirements. Rows with none of the three indicators form the reference category. We keep that label general because its precise composition requires the source coding.

In [ ]:
laws = df.copy()
laws["requirement"] = "Reference category"
laws.loc[laws["cl7"] == 1, "requirement"] = "7 years"
laws.loc[laws["cl8"] == 1, "requirement"] = "8 years"
laws.loc[laws["cl9"] == 1, "requirement"] = "9+ years"
fig = px.box(laws, x="requirement", y="indEduc", color="requirement", points=False, labels={"requirement": "Required-schooling category", "indEduc": "Cell mean schooling (years)"}, title="Compare schooling across law categories")
fig.show()

**Explore the chart:** Hover over the medians. What differences in state or cohort could accompany these law categories?

**Write before reading on:** Describe one pattern with its unit and comparison group. Then give one possible explanation.

<details>
<summary>Interpretation checkpoint</summary>

The box plots mix states and cohorts. A relationship between laws and schooling is one part of an instrument argument, not proof that laws affect wages only through schooling.

</details>

## Visual investigation 3 · Put schooling and wages together

**Predict:** What does the schooling–wage pattern look like at the cell level?

Each dot represents a cell, not a person. Color distinguishes census years, which may reveal why pooling them hides important differences.

In [ ]:
cells = df.assign(census=df["year"].astype(str))
fig = px.scatter(cells, x="indEduc", y="lnwkwage", color="census", hover_data=["sob", "yob"], opacity=0.5, labels=labels, title="Cell averages mix education, place, and time")
fig.show()

**Explore the chart:** Toggle census years. Describe whether the broad relationship changes within a wave.

**Write before reading on:** Describe one pattern with its unit and comparison group. Then give one possible explanation.

<details>
<summary>Interpretation checkpoint</summary>

An aggregate association need not equal an individual return to schooling. Place, cohort, and census-year differences remain part of the comparison.

</details>

## Make it yours · 8 minutes

### 1. Modify an example

In the cohort line chart, replace schooling with lnwkwage and relabel the outcome as mean log weekly wages.

Copy that chart cell before editing, so you can compare the original and modified outputs. Say what changed in your question as well as in your code.

### 2. Try a small investigation

Count how many cells are available in each required-schooling category and plot the counts.

Write a prediction first. The optional starter cell below is safe to leave unchanged; later cells do not depend on your answer.

In [ ]:
# Your investigation goes here.
# Describe your expected pattern before writing code.

<details>
<summary>Hint 1 · choose the right evidence</summary>

Use laws, which contains the readable requirement category.

</details>

<details>
<summary>Hint 2 · a Python starting point</summary>

value_counts on requirement gives a count of cells, not a count of people.

</details>

<details>
<summary>Optional worked solution · open after trying</summary>

<pre><code>counts = laws[&quot;requirement&quot;].value_counts().rename_axis(&quot;requirement&quot;).reset_index(name=&quot;cells&quot;)
fig = px.bar(counts, x=&quot;requirement&quot;, y=&quot;cells&quot;, labels={&quot;requirement&quot;: &quot;Requirement category&quot;, &quot;cells&quot;: &quot;Stored cells&quot;})
fig.show()</code></pre>

Unequal numbers of cells remind us that an unweighted category comparison may represent different sets of cohorts and states.

Copy the code into the starter cell to run it.

</details>

## Optional extension · do weights change the picture?

The chapter uses the source sample-line weight. Compare weighted and equal-cell schooling summaries within a census year; neither comparison identifies a law effect.

A **weighted mean** multiplies each value by its weight, adds those products, and divides by the sum of weights. The code below drops rows missing any required value and keeps positive weights. Each group gets its own denominator. This is a descriptive calculation, not an uncertainty estimate or a causal adjustment.

Predict whether weighting will change the comparison. Then compare the two summaries and explain which units receive more influence.

In [ ]:
weighted_data = df.dropna(subset=["indEduc", "year", "weight"]).copy()
weighted_data = weighted_data[weighted_data["weight"] > 0]
weighted_data["product"] = weighted_data["indEduc"] * weighted_data["weight"]
totals = weighted_data.groupby("year")[["product", "weight"]].sum()
weighted_mean = totals["product"] / totals["weight"]
unweighted_mean = weighted_data.groupby("year")["indEduc"].mean()
comparison = pd.DataFrame({"Equal row weights": unweighted_mean, "Source weights": weighted_mean})
display(comparison.round(3))

## What can you now say? · 4 minutes

Return to your initial prediction. Write a four-sentence evidence note:

1. **I expected…** State your prediction.
2. **I observed…** Use a chart, a comparison, and the correct units.
3. **This does not establish…** Name a causal claim the evidence cannot settle.
4. **To learn more, I would need…** Propose an additional measurement or a more credible comparison.

### Carry this question into the chapter

**What differences between states and birth cohorts would need attention before treating a law as a source of causal variation?**

[Continue with Chapter 6: The Wages of Schooling](https://cmg777.github.io/intro2causal/book/_book/notebooks_quarto/06-wages-of-schooling.html#strategy-3-child-labor-law-iv) · [Return to the case collection](https://cmg777.github.io/intro2causal/data-dictionary.html#ch6-childlabor-clean-csv)

### Source and preparation notes

[Acemoglu and Angrist (2000), How Large Are Human-Capital Externalities? Evidence from Compulsory Schooling Laws; author archive](https://economics.mit.edu/people/faculty/josh-angrist/angrist-data-archive)

Cell-level analysis file used by the notebook; this CSV is already aggregated and its upstream creation is not included in the repository. `weight`/cell sizes are distinct from the unweighted summaries shown.

These exploratory charts are based on the committed CSV. They are not a complete replication of the original study. Keep the source study, this prepared extract, and your descriptive results distinct.